## Objective

The objective of this notebook is to clean and transform the raw Online Retail dataset into a high-quality, analysis-ready dataset.

Based on the findings from the Data Understanding phase, this notebook performs the following tasks:

- Removes duplicate records.
- Handles missing values.
- Removes cancelled transactions.
- Removes invalid transaction records.
- Standardises data types.
- Engineers new features required for downstream analysis.
- Validates the cleaned dataset.
- Exports the final cleaned dataset as the project's shared data source.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Load Raw Dataset




In [2]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Interpretation

The dataset has been successfully loaded into memory and is ready for preprocessing.

At this stage, no modifications have been applied, ensuring that the original data remains intact for reproducibility.

## 3. Initial Dataset Overview

Before applying any preprocessing operations, the dimensions of the dataset are recorded.

These values serve as a baseline for measuring the impact of each cleaning step.



In [3]:
print("Initial Shape:", df.shape)

Initial Shape: (541909, 8)


### Interpretation

The initial dataset dimensions provide a reference point for evaluating how many records are removed during each cleaning stage.

## 4. Remove Duplicate Records

Duplicate records may cause transactions to be counted multiple times, leading to inaccurate sales and customer analyses.

These records are removed to ensure that every transaction is represented only once.

In [6]:
df = df.drop_duplicates()

duplicates = df.duplicated().sum()

print("Removed duplicates:", duplicates)
print("Dataset Shape:", df.shape)

Removed duplicates: 0
Dataset Shape: (536641, 8)


### Interpretation

Duplicate records were successfully removed from the dataset.

Removing duplicates improves data integrity by ensuring that each transaction contributes only once to subsequent analyses.

## 5. Handle Missing Values

The Data Understanding phase identified missing values primarily in the **CustomerID** and **Description** columns.

Customer segmentation requires customer identifiers, while product descriptions are necessary for interpreting purchased items.

Records containing missing values in these columns are removed.

In [7]:
#remove missing customerID

df = df.dropna(subset=["CustomerID"])

In [8]:
#remove missing description

df = df.dropna(subset=["Description"])

In [9]:
print("Missing Values After Cleaning")

print(df.isnull().sum())

print("Dataset Shape:", df.shape)

Missing Values After Cleaning
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64
Dataset Shape: (401604, 8)


### Interpretation

Records containing missing customer identifiers and product descriptions were removed.

This ensures that every remaining transaction can be linked to a customer and a valid product, supporting reliable customer segmentation and purchasing analyses.

## 6. Remove Cancelled Transactions

Invoices beginning with the letter **"C"** represent cancelled orders.

Since these transactions do not correspond to completed purchases, they are excluded from the cleaned dataset.

In [11]:
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
cancelled_transaction = df["InvoiceNo"].astype(str).str.startswith("C").sum()
print("Cancelled Transactions After Cleaning:", cancelled_transaction)
print("Dataset Shape:", df.shape)

Cancelled Transactions After Cleaning: 0
Dataset Shape: (392732, 8)


### Interpretation

Cancelled transactions were successfully removed.

The cleaned dataset now contains only completed customer purchases, making it more appropriate for customer behaviour analysis.

## 7. Remove Invalid Transactions

Negative quantities generally represent returned products or inventory adjustments.

Since the objective of this project is to analyse completed purchases, these records are removed.

In [12]:
df = df[df["Quantity"] > 0]
negative_qt = (df["Quantity"] < 0).sum()
print("Negative Quantities After Cleaning:", negative_qt)
print("Dataset Shape:", df.shape)

Negative Quantities After Cleaning: 0
Dataset Shape: (392732, 8)


### Interpretation

Transactions with negative quantities were removed, ensuring that the dataset reflects only completed purchase transactions.

## 8. Handle Invalid Prices

Unit prices are expected to be positive.

Transactions containing zero or negative unit prices are removed because they do not represent valid sales.

In [14]:
df = df[df["UnitPrice"] > 0]
invalid_prices = (df["UnitPrice"] <= 0).sum()
print("Invalid Prices After Cleaning:", invalid_prices)
print("Dataset Shape:", df.shape)

Invalid Prices After Cleaning: 0
Dataset Shape: (392692, 8)


### Interpretation

Transactions with zero or negative prices were removed to ensure that all remaining transactions represent valid sales values.

## 9. Standardise the Date Format

The **InvoiceDate** column is explicitly converted to the datetime format to ensure consistency and support future time-based analyses.

In [15]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceDate"].dtype

dtype('<M8[ns]')

### Interpretation

The **InvoiceDate** column is now stored as a datetime variable, ensuring consistency for any future temporal analyses.

## 10. Feature Engineering

A new variable named `TotalPrice` is created to represent the total monetary value of each transaction line.

The feature is calculated as:

**TotalPrice = Quantity × UnitPrice**

This feature will support downstream analyses such as revenue calculations and customer value assessment.

In [16]:
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


### Interpretation

The **TotalPrice** feature has been successfully created and represents the monetary value contributed by each purchased product line.

## 11. Outlier Assessment


In retail transaction datasets, unusually large quantities or high prices may represent:

- Legitimate bulk purchases.
- Premium or high-value products.
- Data entry errors.

Before deciding whether to remove these observations, they are identified using the Interquartile Range (IQR) method.

Rather than removing all outliers automatically, each case should be evaluated based on business context to avoid excluding genuine customer purchases.

In [17]:
# Calculate the IQR for Quantity

Q1_quantity = df["Quantity"].quantile(0.25)
Q3_quantity = df["Quantity"].quantile(0.75)

IQR_quantity = Q3_quantity - Q1_quantity

lower_quantity = Q1_quantity - 1.5 * IQR_quantity
upper_quantity = Q3_quantity + 1.5 * IQR_quantity

quantity_outliers = df[
    (df["Quantity"] < lower_quantity) |
    (df["Quantity"] > upper_quantity)
]

print("Quantity Outliers:", len(quantity_outliers))

quantity_outliers.head()

Quantity Outliers: 25616


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,54.08
31,536370,10002,INFLATABLE POLITICAL GLOBE,48,2010-12-01 08:45:00,0.85,12583.0,France,40.80
44,536370,22492,MINI PAINT SET VINTAGE,36,2010-12-01 08:45:00,0.65,12583.0,France,23.40
46,536371,22086,PAPER CHAIN KIT 50'S CHRISTMAS,80,2010-12-01 09:00:00,2.55,13748.0,United Kingdom,204.00
65,536374,21258,VICTORIAN SEWING BOX LARGE,32,2010-12-01 09:09:00,10.95,15100.0,United Kingdom,350.40


### Interpretation

The IQR method identifies transactions with unusually small or large purchase quantities.

Large quantities are common in wholesale and online retail environments because customers may purchase products in bulk.

Therefore, these observations are not automatically considered errors.

In [18]:
#unitprice outliers

Q1_price = df["UnitPrice"].quantile(0.25)

Q3_price = df["UnitPrice"].quantile(0.75)

IQR_price = Q3_price - Q1_price

lower_price = Q1_price - 1.5 * IQR_price
upper_price = Q3_price + 1.5 * IQR_price

price_outliers = df[
    (df["UnitPrice"] < lower_price) |
    (df["UnitPrice"] > upper_price)
]

print("Price Outliers:", len(price_outliers))

price_outliers.head()

Price Outliers: 34112


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.3
16,536367,22622,BOX OF VINTAGE ALPHABET BLOCKS,2,2010-12-01 08:34:00,9.95,13047.0,United Kingdom,19.9
19,536367,21777,RECIPE BOX WITH METAL HEART,4,2010-12-01 08:34:00,7.95,13047.0,United Kingdom,31.8
20,536367,48187,DOORMAT NEW ENGLAND,4,2010-12-01 08:34:00,7.95,13047.0,United Kingdom,31.8
45,536370,POST,POSTAGE,3,2010-12-01 08:45:00,18.00,12583.0,France,54.0


### Interpretation

The analysis identifies transactions with unusually high or low unit prices.

While some extreme prices may represent data entry errors, others may correspond to premium products sold by the retailer.

Consequently, these observations require business context before removal.

## 11. Validate the Cleaned Dataset

The cleaned dataset is validated to confirm that all preprocessing steps have been successfully applied.

In [20]:
print("Final Dataset Shape:", df.shape)

print("\nMissing Values")

print(df.isnull().sum())

print("\nDuplicate Rows")

print(df.duplicated().sum())

print("\nNegative Quantities")

print((df["Quantity"] < 0).sum())

print("\nInvalid Prices")

print((df["UnitPrice"] <= 0).sum())

print("\nCancelled Transactions")

print(df["InvoiceNo"].astype(str).str.startswith("C").sum())

Final Dataset Shape: (392692, 9)

Missing Values
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

Duplicate Rows
0

Negative Quantities
0

Invalid Prices
0

Cancelled Transactions
0


In [19]:
#decision

print("Quantity Outliers:", len(quantity_outliers))

print("Price Outliers:", len(price_outliers))

Quantity Outliers: 25616
Price Outliers: 34112


### Interpretation

The validation confirms that the cleaning pipeline has been successfully applied.

The dataset no longer contains duplicate records, missing customer identifiers, cancelled transactions, negative quantities, or invalid prices, making it suitable for downstream analysis.

### Decision

The identified outliers were reviewed but **were not removed** from the dataset.

This decision was made because:

- The project documentation does not define business rules for acceptable purchase quantities or prices.
- Large purchases are common in retail and may represent genuine customer behaviour.
- Removing these observations could eliminate valuable information required for customer segmentation.

Therefore, the outliers were retained in the cleaned dataset.

## 12. Export the Clean Dataset



In [21]:
output_path = "../data/processed/online_retail_clean.csv"

df.to_csv(output_path, index=False)

print(f"Dataset successfully saved to: {output_path}")

Dataset successfully saved to: ../data/processed/online_retail_clean.csv


## 14. Conclusion

The Data Preparation phase transformed the raw Online Retail dataset into a clean, consistent, and analysis-ready dataset suitable for customer segmentation and downstream analysis.

The preprocessing pipeline successfully addressed the major data quality issues identified during the Data Understanding phase by:

- Removing duplicate records.
- Handling missing values.
- Excluding cancelled transactions.
- Removing returned products represented by negative quantities.
- Removing transactions with invalid pricing.
- Standardising date formats.
- Engineering the **TotalPrice** feature.

Additionally, potential outliers were identified using the Interquartile Range (IQR) method. Rather than removing them automatically, they were retained because no business rules indicated that these observations were invalid, and they may represent legitimate high-value customer purchases.

Finally, the cleaned dataset was exported as **data/processed/online_retail_clean.csv**, which serves as the project's single source of truth for all subsequent CRISP-DM phases, ensuring consistency across Exploratory Data Analysis, Modelling, Evaluation, and Reporting.